# Data Preprocess - Class Distribution (Model2)

Use this notebook to inspect class imbalance on:
- Source YOLO labels (`Dataset-v3.v1i.yolov5pytorch/train/labels`)
- Processed COCO dataset (`data/rfdetr_tiled_coco`) if generated

In [3]:
from pathlib import Path
from collections import Counter
import json

source_root = Path("Dataset-v3.v1i.yolov5pytorch")
label_dir = source_root / "train" / "labels"
data_yaml = source_root / "data.yaml"

if not label_dir.exists():
    raise FileNotFoundError(f"Label directory not found: {label_dir.resolve()}")

# class names from data.yaml (fallback to class_{id})
class_name_map = {}
if data_yaml.exists():
    try:
        import yaml
        payload = yaml.safe_load(data_yaml.read_text(encoding="utf-8"))
        names = payload.get("names", [])
        if isinstance(names, list):
            class_name_map = {i: str(v) for i, v in enumerate(names)}
        elif isinstance(names, dict):
            class_name_map = {int(k): str(v) for k, v in names.items()}
    except Exception as exc:
        print(f"[WARN] Failed to parse data.yaml: {exc}")

bbox_counter = Counter()
file_counter = Counter()
invalid_line_count = 0

label_files = sorted(label_dir.glob("*.txt"))
for txt_path in label_files:
    classes_in_file = set()
    for raw in txt_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 5:
            invalid_line_count += 1
            continue
        try:
            cls = int(float(parts[0]))
        except ValueError:
            invalid_line_count += 1
            continue
        bbox_counter[cls] += 1
        classes_in_file.add(cls)
    for cls in classes_in_file:
        file_counter[cls] += 1

total_bbox = sum(bbox_counter.values())
rows = []
for cls in sorted(bbox_counter):
    rows.append({
        "class_id": cls,
        "class_name": class_name_map.get(cls, f"class_{cls}"),
        "bbox_count": bbox_counter[cls],
        "files_containing_class": file_counter[cls],
        "bbox_ratio(%)": round((bbox_counter[cls] / total_bbox) * 100, 4) if total_bbox else 0.0,
    })

try:
    import pandas as pd
    df = pd.DataFrame(rows).sort_values("bbox_count", ascending=False)
    display(df)
except Exception:
    print("class_id\tclass_name\tbbox_count\tfiles_containing_class\tbbox_ratio(%)")
    for r in sorted(rows, key=lambda x: x["bbox_count"], reverse=True):
        print(f"{r['class_id']}\t{r['class_name']}\t{r['bbox_count']}\t{r['files_containing_class']}\t{r['bbox_ratio(%)']}")

print("\n[Source Summary]")
print(f"label files: {len(label_files)}")
print(f"total bboxes: {total_bbox}")
print(f"invalid lines: {invalid_line_count}")
if bbox_counter:
    mx = max(bbox_counter.values())
    mn = min(v for v in bbox_counter.values() if v > 0)
    print(f"imbalance ratio (max/min): {mx}/{mn} = {mx/mn:.3f}")


,class_id,class_name,bbox_count,files_containing_class,bbox_ratio(%)
5,5,pockmark,10643,2025,51.9475
4,4,gasbubble,3146,1464,15.3553
1,1,blackspot,2734,1675,13.3444
7,7,unknown,1649,637,8.0486
3,3,dust,817,385,3.9877
0,0,airbubble,731,417,3.5679
6,6,scratch,540,381,2.6357
2,2,color-distribution,228,142,1.1128



[Source Summary]
label files: 3077
total bboxes: 20488
invalid lines: 0
imbalance ratio (max/min): 10643/228 = 46.680


In [4]:
from pathlib import Path
from collections import Counter
import json

coco_root = Path("data/rfdetr_tiled_coco")
splits = ["train", "valid", "test"]

if not coco_root.exists():
    print(f"Processed dataset not found: {coco_root.resolve()}")
    print("Run scripts/prepare_tiled_coco_dataset.py first.")
else:
    per_split = {}
    overall = Counter()
    category_name = {}

    for split in splits:
        ann_path = coco_root / split / "_annotations.coco.json"
        if not ann_path.exists():
            print(f"[WARN] Missing annotation: {ann_path}")
            continue

        payload = json.loads(ann_path.read_text(encoding="utf-8"))
        cats = payload.get("categories", [])
        anns = payload.get("annotations", [])
        imgs = payload.get("images", [])

        if not category_name:
            category_name = {int(c["id"]): str(c["name"]) for c in cats}

        c = Counter(int(a["category_id"]) for a in anns)
        per_split[split] = {
            "images": len(imgs),
            "annotations": len(anns),
            "class_counter": c,
        }
        overall.update(c)

    print("[Processed COCO Summary]")
    for split in splits:
        if split not in per_split:
            continue
        print(f"- {split}: images={per_split[split]['images']}, annotations={per_split[split]['annotations']}")

    rows = []
    total = sum(overall.values())
    for cat_id in sorted(overall):
        row = {
            "category_id": cat_id,
            "class_name": category_name.get(cat_id, f"cat_{cat_id}"),
            "total_bbox": overall[cat_id],
            "total_ratio(%)": round((overall[cat_id] / total) * 100, 4) if total else 0.0,
        }
        for split in splits:
            row[f"{split}_bbox"] = per_split.get(split, {}).get("class_counter", Counter()).get(cat_id, 0)
        rows.append(row)

    try:
        import pandas as pd
        display(pd.DataFrame(rows).sort_values("total_bbox", ascending=False))
    except Exception:
        print("category_id\tclass_name\ttotal_bbox\ttotal_ratio(%)\ttrain\tvalid\ttest")
        for r in sorted(rows, key=lambda x: x["total_bbox"], reverse=True):
            print(
                f"{r['category_id']}\t{r['class_name']}\t{r['total_bbox']}\t{r['total_ratio(%)']}\t"
                f"{r['train_bbox']}\t{r['valid_bbox']}\t{r['test_bbox']}"
            )

    print("\nTip: compare split-wise class ratios to check if split strategy kept distribution stable.")


[Processed COCO Summary]
- train: images=9162, annotations=16137
- valid: images=1796, annotations=3209
- test: images=1206, annotations=2195


,category_id,class_name,total_bbox,total_ratio(%),train_bbox,valid_bbox,test_bbox
5,6,pockmark,10873,50.4758,8132,1611,1130
4,5,gasbubble,3334,15.4775,2545,458,331
1,2,blackspot,2783,12.9195,2090,395,298
7,8,unknown,1712,7.9476,1277,264,171
3,4,dust,928,4.3081,698,168,62
0,1,airbubble,760,3.5282,554,125,81
6,7,scratch,743,3.4492,557,110,76
2,3,color-distribution,408,1.8941,284,78,46



Tip: compare split-wise class ratios to check if split strategy kept distribution stable.
